# 07 — Feature selection

**Goal:** keep only the most informative engineered features (mutual information).

## Why this experiment?
More features are not always better. Noise can hurt or add training cost.
We test whether selecting ~18 best features helps or hurts ROC-AUC.

## Approach
1. Build the full engineered feature set.
2. Inside a pipeline: impute → SelectKBest(mutual_info) → LightGBM.
3. Selection is fit on train only (no test leakage).

## What changed?
- Same full FE as 06 (without the extra interaction/bin layer)
- Added mutual-information feature pruning before the model

## Features used in this notebook
- Start from full pricing + time + geo features
- **Selection method:** SelectKBest + mutual information → keep top ~18
- **How selected:** keep features that share the most information with `hyper_ack` on **train only**
- **Why:** test whether fewer features are enough (here, full set still wins overall)


### Setup
Shared data load and fixed split.


In [ ]:
import os
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "shared" / "protocol.py").exists():
        EXPERIMENT_ROOT = candidate
        break
    if (candidate / "hyperack_exp" / "shared" / "protocol.py").exists():
        EXPERIMENT_ROOT = candidate / "hyperack_exp"
        break
else:
    raise RuntimeError("Run this notebook from the HyperAck project directory.")
os.chdir(EXPERIMENT_ROOT)
sys.path.insert(0, str(EXPERIMENT_ROOT))

from shared.protocol import (
    add_geo_features,
    add_pricing_features,
    add_time_features,
    base_features,
    evaluate,
    load_clean_df,
    make_xy,
    save_result,
    split_frame,
    actual_vs_predicted_report,
)

RANDOM_STATE = 42
train_df, test_df = split_frame(load_clean_df())


### Features
Full engineered matrix before selection.


In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif
X_train, y_train = make_xy(train_df, pricing=True, time_features=True, geo=True)
X_test, y_test = make_xy(test_df, pricing=True, time_features=True, geo=True)


### Candidate features (before selection)

We first build the full engineered matrix. The model pipeline then keeps only the top features by **mutual information** with the target.

| Step | What happens |
|------|----------------|
| 1 | Build pricing + time + geo features |
| 2 | Impute missing values on train |
| 3 | SelectKBest(mutual_info) keeps ~18 columns |
| 4 | LightGBM trains on the reduced set |

Next cell shows the **candidate** columns. After training, a later cell shows which ones were actually kept.


In [ ]:
feature_why = {
    "deliverey_category_id": "Delivery type — some categories get accepted more often",
    "weekday": "Day of week — weekday vs weekend courier behavior",
    "time_bucket": "Coarse time-of-day bucket from the raw data",
    "total_distance": "Trip length — longer trips can be harder to accept",
    "sum_product": "Order size / number of products",
    "source_latitude": "Pickup latitude — area effects",
    "source_longitude": "Pickup longitude — area effects",
    "destination_latitude": "Drop-off latitude — area effects",
    "destination_longitude": "Drop-off longitude — area effects",
    "first_customer_fare": "First offered customer price (usually known early)",
    "final_customer_fare": "Final customer price — strong but may be post-decision",
    "final_biker_fare": "Final courier pay — strong but may be post-decision",
    "geo_cluster": "Train-only KMeans region of the trip (pickup+drop-off)",
    "log_distance": "Log distance — softens very long trips",
    "first_fare_per_km": "First fare ÷ distance — pay vs effort",
    "final_customer_fare_per_km": "Final customer fare ÷ distance",
    "customer_fare_delta": "Final − first customer fare (price change)",
    "customer_fare_change_pct": "Relative fare change vs first offer",
    "biker_customer_gap": "Biker fare − customer fare (split / margin)",
    "biker_fare_per_km": "Courier pay per km",
    "log_final_customer_fare": "Log of final customer fare",
    "log_final_biker_fare": "Log of final biker fare",
    "hour": "Exact hour of order creation",
    "is_rush_hour": "Lunch/evening peak flag",
    "is_weekend": "Weekend flag",
    "hour_sin": "Cyclical hour (sin) so 23 is near 0",
    "hour_cos": "Cyclical hour (cos)",
    "weekday_sin": "Cyclical weekday (sin)",
    "weekday_cos": "Cyclical weekday (cos)",
    "day_of_month": "Calendar day — mild monthly pattern",
    "haversine_km": "Great-circle route distance in km",
    "latitude_delta": "North/south trip span",
    "longitude_delta": "East/west trip span",
    "geo_bearing_sin": "Trip direction (sin of bearing)",
    "geo_bearing_cos": "Trip direction (cos of bearing)",
    "distance_x_first_fare": "Interaction: long trip × price",
    "category_x_hour": "Interaction: category × hour",
    "total_distance_qbin": "Train-fitted distance quantile bin",
    "first_customer_fare_qbin": "Train-fitted first-fare quantile bin"
}

cols = list(X_train.columns)
rows = []
for c in cols:
    rows.append({
        "feature": c,
        "why_selected": feature_why.get(c, "Part of this experiment's engineered feature set"),
    })
feature_table = pd.DataFrame(rows)
print(f"Total features selected: {len(cols)}")
print("Columns:")
print(", ".join(cols))
feature_table


### Model pipeline
SelectKBest then LightGBM.


In [ ]:
from lightgbm import LGBMClassifier
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("select", SelectKBest(mutual_info_classif, k=min(18, X_train.shape[1]))),
    ("model", LGBMClassifier(n_estimators=800, learning_rate=0.035, num_leaves=31, reg_lambda=1.0, random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1)),
])


### Evaluate
Held-out metrics.


In [ ]:
metrics = evaluate(model, X_train, y_train, X_test, y_test)
metrics


### Actual vs predicted (test set)

After training, we score the **held-out test set** and compare:

1. **Actual** labels (`hyper_ack`) vs **predicted** labels  
2. Confusion matrix (rows = actual, columns = predicted)  
3. Per-class precision / recall / F1  
4. A sample of correct and incorrect rows with predicted probability  

This is only test-set performance — not training rows.


In [ ]:
from IPython.display import display
from shared.protocol import actual_vs_predicted_report

avp = actual_vs_predicted_report(
    metrics["y_true"],
    metrics["y_pred"],
    metrics["y_prob"],
    sample_size=25,
)
print("1) Actual vs predicted class counts")
display(avp["class_counts"])
print("2) Confusion matrix")
display(avp["confusion_matrix"])
print("3) Outcome breakdown")
display(avp["outcomes"])
print("4) Per-class metrics")
display(avp["per_class_metrics"])
print("5) Sample of actual vs predicted rows")
display(avp["prediction_sample"])


### Features actually kept by SelectKBest

Mutual information ranks how much each candidate tells us about `hyper_ack`. Only the top-k survive into LightGBM.


In [ ]:
# Recover selected feature names from the fitted pipeline
selector = model.named_steps["select"]
imputer = model.named_steps["imputer"]
support = selector.get_support()
# columns after imputer keep original order
candidate_cols = list(X_train.columns)
selected = [c for c, keep in zip(candidate_cols, support) if keep]
scores = selector.scores_
score_map = {c: float(s) for c, s in zip(candidate_cols, scores)}
feature_why = {
    "deliverey_category_id": "Delivery type — some categories get accepted more often",
    "weekday": "Day of week — weekday vs weekend courier behavior",
    "time_bucket": "Coarse time-of-day bucket from the raw data",
    "total_distance": "Trip length — longer trips can be harder to accept",
    "sum_product": "Order size / number of products",
    "source_latitude": "Pickup latitude — area effects",
    "source_longitude": "Pickup longitude — area effects",
    "destination_latitude": "Drop-off latitude — area effects",
    "destination_longitude": "Drop-off longitude — area effects",
    "first_customer_fare": "First offered customer price (usually known early)",
    "final_customer_fare": "Final customer price — strong but may be post-decision",
    "final_biker_fare": "Final courier pay — strong but may be post-decision",
    "geo_cluster": "Train-only KMeans region of the trip (pickup+drop-off)",
    "log_distance": "Log distance — softens very long trips",
    "first_fare_per_km": "First fare ÷ distance — pay vs effort",
    "final_customer_fare_per_km": "Final customer fare ÷ distance",
    "customer_fare_delta": "Final − first customer fare (price change)",
    "customer_fare_change_pct": "Relative fare change vs first offer",
    "biker_customer_gap": "Biker fare − customer fare (split / margin)",
    "biker_fare_per_km": "Courier pay per km",
    "log_final_customer_fare": "Log of final customer fare",
    "log_final_biker_fare": "Log of final biker fare",
    "hour": "Exact hour of order creation",
    "is_rush_hour": "Lunch/evening peak flag",
    "is_weekend": "Weekend flag",
    "hour_sin": "Cyclical hour (sin) so 23 is near 0",
    "hour_cos": "Cyclical hour (cos)",
    "weekday_sin": "Cyclical weekday (sin)",
    "weekday_cos": "Cyclical weekday (cos)",
    "day_of_month": "Calendar day — mild monthly pattern",
    "haversine_km": "Great-circle route distance in km",
    "latitude_delta": "North/south trip span",
    "longitude_delta": "East/west trip span",
    "geo_bearing_sin": "Trip direction (sin of bearing)",
    "geo_bearing_cos": "Trip direction (cos of bearing)",
    "distance_x_first_fare": "Interaction: long trip × price",
    "category_x_hour": "Interaction: category × hour",
    "total_distance_qbin": "Train-fitted distance quantile bin",
    "first_customer_fare_qbin": "Train-fitted first-fare quantile bin"
}
selected_table = pd.DataFrame([
    {
        "feature": c,
        "mutual_info_score": round(score_map[c], 4),
        "why_it_matters": feature_why.get(c, "Selected by mutual information vs hyper_ack"),
    }
    for c in selected
]).sort_values("mutual_info_score", ascending=False).reset_index(drop=True)
print(f"Selected {len(selected)} / {len(candidate_cols)} features")
selected_table


### Save result
Write experiment `07`.


In [ ]:
result_path = save_result(
    "07",
    "07_feature_selection",
    "Mutual-information selection followed by LightGBM",
    metrics,
    best_model="SelectKBest + LGBMClassifier",
    notes="Selection is fit exclusively on training data inside the pipeline.",
    feature_count=X_train.shape[1],
)
pd.Series(metrics).drop("confusion_matrix").sort_index(), result_path


## What to look at
- If ROC-AUC stays close to 06 with fewer features → selection is useful.
- If it drops a lot → keep the full feature set.
